In [2]:
!pip install torch torchvision transformers tifffile numpy pandas scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.2/111.2 MB 10.2 MB/s  0:00:10m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 19.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 17.2 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 770.3/770.3 kB 15.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 7.8 MB/s  0:00:00 eta 0:00:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 20.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 18.6 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.3/8.3 MB 10.2 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 21.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.3/20.3 MB 14.5 MB/s  0:00:01 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 4.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 20.0 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━

In [7]:
import os
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"
import os
import numpy as np
import pandas as pd
import tifffile
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import SegformerForSemanticSegmentation

In [8]:
BASE_DIR = "/Users/shaurya/Downloads/PHY:MATH A LVL TOPICALS/India/Mumbai/data"
TILES_DIR = os.path.join(BASE_DIR, "tiles", "tiles_out")
TILE_INDEX_PATH = os.path.join(TILES_DIR, "tile_index.csv")

tiles = pd.read_csv(TILE_INDEX_PATH)

max_col = tiles["col"].max()
val_cols = set(range(max_col - 1, max_col + 1))  # last 2 columns = validation strip

train_df = tiles[~tiles["col"].isin(val_cols)].reset_index(drop=True)
val_df = tiles[tiles["col"].isin(val_cols)].reset_index(drop=True)

print(f"Train tiles: {len(train_df)}, Val tiles: {len(val_df)}")
print(f"Train slum tiles: {(train_df.slum_fraction > 0).sum()}, "
      f"Val slum tiles: {(val_df.slum_fraction > 0).sum()}")

Train tiles: 75, Val tiles: 30
Train slum tiles: 56, Val slum tiles: 13


In [9]:
class SlumDataset(Dataset):
    def __init__(self, df, tiles_dir):
        self.df = df
        self.tiles_dir = tiles_dir

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        path = os.path.join(self.tiles_dir, row["filename"])

        arr = tifffile.imread(path)  # shape: (11, 256, 256)
        image = arr[:10].astype(np.float32)
        mask = arr[10].astype(np.int64)  # 0 or 1

        # Normalize reflectance bands (values roughly 0-20000) to 0-1
        image = np.clip(image / 10000.0, 0, 1)

        image = torch.from_numpy(image)
        mask = torch.from_numpy(mask)
        return image, mask


train_ds = SlumDataset(train_df, TILES_DIR)
val_ds = SlumDataset(val_df, TILES_DIR)

train_loader = DataLoader(train_ds, batch_size=4, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=4, shuffle=False)

# quick sanity check
img, msk = train_ds[0]
print(img.shape, msk.shape, msk.unique())

torch.Size([10, 256, 256]) torch.Size([256, 256]) tensor([0])


In [11]:
print(model)

SegformerForSemanticSegmentation(
  (segformer): SegformerModel(
    (stages): ModuleList(
      (0): SegformerStage(
        (patch_embeddings): SegformerOverlapPatchEmbeddings(
          (proj): Conv2d(3, 32, kernel_size=(7, 7), stride=(4, 4), padding=(3, 3))
          (layer_norm): LayerNorm((32,), eps=1e-05, elementwise_affine=True, bias=True)
        )
        (blocks): ModuleList(
          (0): SegformerLayer(
            (layernorm_before): LayerNorm((32,), eps=1e-05, elementwise_affine=True, bias=True)
            (attention): SegformerAttention(
              (q_proj): Linear(in_features=32, out_features=32, bias=True)
              (k_proj): Linear(in_features=32, out_features=32, bias=True)
              (v_proj): Linear(in_features=32, out_features=32, bias=True)
              (o_proj): Linear(in_features=32, out_features=32, bias=True)
              (sequence_reduction): SegformerSequenceReduction(
                (sequence_reduction): Conv2d(32, 32, kernel_size=(8, 8), s

In [17]:
model = SegformerForSemanticSegmentation.from_pretrained(
    "nvidia/mit-b0",
    num_labels=2,               # slum vs not-slum
    ignore_mismatched_sizes=True,
)

# Swap the first conv layer to accept 10 bands instead of 3
old_conv = model.segformer.stages[0].patch_embeddings.proj
new_conv = torch.nn.Conv2d(
    in_channels=10,
    out_channels=old_conv.out_channels,
    kernel_size=old_conv.kernel_size,
    stride=old_conv.stride,
    padding=old_conv.padding,
)

# Keep the pretrained weights for the first 3 channels, random-init the rest
with torch.no_grad():
    new_conv.weight[:, :3] = old_conv.weight
    torch.nn.init.kaiming_normal_(new_conv.weight[:, 3:])
    new_conv.bias.copy_(old_conv.bias)

model.segformer.stages[0].patch_embeddings.proj = new_conv

device = torch.device("cpu")
model.to(device)
print(f"Using device: {device}")

[transformers] You passed `num_labels=2` which is incompatible to the `id2label` map of length `1000`.
Loading weights: 100%|██████████| 192/192 [00:00<00:00, 47894.99it/s]
[transformers] SegformerForSemanticSegmentation LOAD REPORT from: nvidia/mit-b0
Key                                                     | Status     | 
--------------------------------------------------------+------------+-
classifier.bias                                         | UNEXPECTED | 
classifier.weight                                       | UNEXPECTED | 
decode_head.batch_norm.num_batches_tracked              | MISSING    | 
decode_head.linear_projections.{0, 1, 2, 3}.proj.weight | MISSING    | 
decode_head.classifier.bias                             | MISSING    | 
decode_head.classifier.weight                           | MISSING    | 
decode_head.linear_projections.{0, 1, 2, 3}.proj.bias   | MISSING    | 
decode_head.linear_fuse.weight                          | MISSING    | 
decode_head.batch_norm.bias

Using device: cpu


In [22]:
import torch.nn as nn
import torch.nn.functional as F

# Weighted so the model can't just lazily predict "no slum" everywhere.
# Slum pixels are ~4-5% of all pixels, so we weight that class higher.
class_weights = torch.tensor([1.0, 5.0]).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights)

In [23]:
optimizer = torch.optim.AdamW(model.parameters(), lr=6e-5)

In [24]:
num_epochs = 20

for epoch in range(num_epochs):
    model.train()
    total_loss = 0

    for images, masks in train_loader:
        images = images.to(device).contiguous()
        masks = masks.to(device)

        optimizer.zero_grad()

        outputs = model(pixel_values=images)
        logits = outputs.logits  # shape: (batch, 2, H/4, W/4) - SegFormer downsamples

        # Upsample logits back to full 256x256 to match the mask size
        logits = F.interpolate(logits, size=masks.shape[-2:], mode="bilinear", align_corners=False)

        loss = criterion(logits, masks)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_train_loss = total_loss / len(train_loader)

    # Quick validation pass
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for images, masks in val_loader:
            images = images.to(device).contiguous()
            masks = masks.to(device)
            outputs = model(pixel_values=images)
            logits = F.interpolate(outputs.logits, size=masks.shape[-2:], mode="bilinear", align_corners=False)
            loss = criterion(logits, masks)
            val_loss += loss.item()
    avg_val_loss = val_loss / len(val_loader)

    print(f"Epoch {epoch+1}/{num_epochs} - train loss: {avg_train_loss:.4f} - val loss: {avg_val_loss:.4f}")

Epoch 1/20 - train loss: 0.2283 - val loss: 0.1520
Epoch 2/20 - train loss: 0.2160 - val loss: 0.1210
Epoch 3/20 - train loss: 0.2110 - val loss: 0.1246
Epoch 4/20 - train loss: 0.2069 - val loss: 0.1266
Epoch 5/20 - train loss: 0.2021 - val loss: 0.1180
Epoch 6/20 - train loss: 0.1907 - val loss: 0.1244
Epoch 7/20 - train loss: 0.1951 - val loss: 0.1279
Epoch 8/20 - train loss: 0.1858 - val loss: 0.1235
Epoch 9/20 - train loss: 0.1842 - val loss: 0.1205
Epoch 10/20 - train loss: 0.1806 - val loss: 0.1254
Epoch 11/20 - train loss: 0.1812 - val loss: 0.1154
Epoch 12/20 - train loss: 0.1745 - val loss: 0.1212
Epoch 13/20 - train loss: 0.1834 - val loss: 0.1182
Epoch 14/20 - train loss: 0.1740 - val loss: 0.1127
Epoch 15/20 - train loss: 0.1696 - val loss: 0.1164
Epoch 16/20 - train loss: 0.1713 - val loss: 0.1100
Epoch 17/20 - train loss: 0.1657 - val loss: 0.1306
Epoch 18/20 - train loss: 0.1639 - val loss: 0.1304
Epoch 19/20 - train loss: 0.1589 - val loss: 0.1313
Epoch 20/20 - train l

In [25]:
model.eval()

total_correct = 0
total_pixels = 0
intersection = 0
union = 0
true_positive = 0
false_positive = 0
false_negative = 0

with torch.no_grad():
    for images, masks in val_loader:
        images = images.to(device)
        masks = masks.to(device)

        outputs = model(pixel_values=images)
        logits = F.interpolate(outputs.logits, size=masks.shape[-2:], mode="bilinear", align_corners=False)
        preds = torch.argmax(logits, dim=1)  # 0 or 1 per pixel

        total_correct += (preds == masks).sum().item()
        total_pixels += masks.numel()

        pred_slum = (preds == 1)
        true_slum = (masks == 1)

        intersection += (pred_slum & true_slum).sum().item()
        union += (pred_slum | true_slum).sum().item()
        true_positive += (pred_slum & true_slum).sum().item()
        false_positive += (pred_slum & ~true_slum).sum().item()
        false_negative += (~pred_slum & true_slum).sum().item()

pixel_accuracy = total_correct / total_pixels
iou_slum = intersection / union if union > 0 else 0
precision = true_positive / (true_positive + false_positive) if (true_positive + false_positive) > 0 else 0
recall = true_positive / (true_positive + false_negative) if (true_positive + false_negative) > 0 else 0
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

print(f"Pixel accuracy: {pixel_accuracy:.4f}")
print(f"Slum IoU: {iou_slum:.4f}")
print(f"Slum precision: {precision:.4f}")
print(f"Slum recall: {recall:.4f}")
print(f"Slum F1: {f1:.4f}")

Pixel accuracy: 0.9695
Slum IoU: 0.5101
Slum precision: 0.6096
Slum recall: 0.7576
Slum F1: 0.6756
